# Market Making Simulation: Kalshi NBA Markets

Can we profitably provide liquidity on Kalshi NBA contracts? The order book notebook
showed theoretical spreads exceed maker fees (KXNBAPTS median spread $0.04 vs maker RT
fee ~$0.006), but that ignores **adverse selection** — the tendency for your fills to
cluster in moments when the market is moving against you.

This notebook replays 3 days of live WebSocket data (Apr 18-21, 2026) to simulate
passive market making and measure:

1. **Fill rate**: how often would resting limit orders at best bid/ask actually execute?
2. **Adverse selection**: after a fill, how does the mid-price move against you?
3. **Realized P&L**: spread captured minus adverse selection minus fees
4. **Which series are viable**: KXNBAPTS vs KXNBAGAME vs KXNBASPREAD vs KXNBATOTAL

**Data sources:**
- `materialized/kalshi_ws/trade.parquet` — 1M+ trades from live WS
- `materialized/kalshi_ws/orderbook_snapshot.parquet` — initial book states
- `materialized/kalshi_ws/orderbook_delta.parquet` — 10M+ book updates

**Fee model:** Kalshi NBA series — maker fee = `0.0175 * C * (1-C)` per contract.

In [ ]:
import json
import gzip
import re
import io
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone, timedelta

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

s3 = boto3.client("s3")
S3_BUCKET = "prediction-markets-data"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

# --- Fee model (Kalshi NBA series) ---
def kalshi_taker_fee(price):
    return 0.07 * price * (1.0 - price)

def kalshi_maker_fee(price):
    return 0.25 * kalshi_taker_fee(price)

# --- Book reconstruction utilities ---
MIN_SIZE = 0.5  # cents — phantom level cleanup threshold

def book_stats(yes_bk, no_bk):
    """Compute best bid/ask/spread/depth from YES and NO books."""
    yes_prices = [(float(p), s) for p, s in yes_bk.items() if s > MIN_SIZE]
    no_prices = [(float(p), s) for p, s in no_bk.items() if s > MIN_SIZE]
    if not yes_prices or not no_prices:
        return None, None, None, 0, 0, 0, 0, 0
    best_bid_p, best_bid_sz = max(yes_prices, key=lambda x: x[0])
    best_no_p, best_no_sz = max(no_prices, key=lambda x: x[0])
    best_ask_p = round(1.0 - best_no_p, 4)
    spread = round(best_ask_p - best_bid_p, 4)
    yes_depth = sum(s for _, s in yes_prices)
    no_depth = sum(s for _, s in no_prices)
    return best_bid_p, best_ask_p, spread, yes_depth, no_depth, best_bid_sz, best_no_sz

def apply_delta(book, price_key, delta_size):
    new_size = book.get(price_key, 0.0) + delta_size
    if new_size < MIN_SIZE:
        book.pop(price_key, None)
    else:
        book[price_key] = new_size

## 1. Load data

Load all three materialized Parquet files, tag each ticker with its series, and
show the universe we're working with.

In [ ]:
# --- Load all three data sources ---
print("Loading materialized Parquet files...")

obj = s3.get_object(Bucket=S3_BUCKET, Key="materialized/kalshi_ws/trade.parquet")
trades_df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
trades_df["t_dt"] = pd.to_datetime(trades_df["t_receipt"], unit="s", utc=True)

obj = s3.get_object(Bucket=S3_BUCKET, Key="materialized/kalshi_ws/orderbook_snapshot.parquet")
snap_df = pd.read_parquet(io.BytesIO(obj["Body"].read()))

obj = s3.get_object(Bucket=S3_BUCKET, Key="materialized/kalshi_ws/orderbook_delta.parquet")
delta_df = pd.read_parquet(io.BytesIO(obj["Body"].read()))

print(f"Trades:    {len(trades_df):>12,} rows, {trades_df['ticker'].nunique():>5} tickers")
print(f"Snapshots: {len(snap_df):>12,} rows, {snap_df['ticker'].nunique():>5} tickers")
print(f"Deltas:    {len(delta_df):>12,} rows, {delta_df['ticker'].nunique():>5} tickers")

# Tag each ticker with its series
def get_series(ticker):
    m = re.match(r"(KXNBA\w+?)-", ticker)
    return m.group(1) if m else "unknown"

trades_df["series"] = trades_df["ticker"].apply(get_series)
snap_df["series"] = snap_df["ticker"].apply(get_series)
delta_df["series"] = delta_df["ticker"].apply(get_series)

# Universe summary
print(f"\nTrades by series:")
for series, grp in trades_df.groupby("series"):
    print(f"  {series:15s}: {len(grp):>9,} trades, {grp['ticker'].nunique():>4} tickers")

# Time range
t_min = pd.to_datetime(trades_df["t_receipt"].min(), unit="s", utc=True)
t_max = pd.to_datetime(trades_df["t_receipt"].max(), unit="s", utc=True)
print(f"\nTime range: {t_min} to {t_max}")
print(f"Duration:   {(trades_df['t_receipt'].max() - trades_df['t_receipt'].min()) / 3600:.1f} hours")

## 2. Build snapshot index for multi-session reconstruction

Each WebSocket connection produces one snapshot per ticker. When the ingester reconnects,
we get a fresh snapshot + new delta stream. To reconstruct the full book history, we need
to replay **all** snapshot sessions, not just the last one.

Build a lookup: `ticker → [list of snapshot metas]`, each with its `conn_id` and book state.

In [ ]:
# --- Build snapshot index: ticker → list of snapshot metas ---
all_snaps = {}  # ticker → [{"conn_id", "t_receipt", "yes_book", "no_book"}, ...]
for _, row in snap_df.iterrows():
    if pd.isna(row.get("yes_book_json")) or pd.isna(row.get("no_book_json")):
        continue
    tk = row["ticker"]
    if tk not in all_snaps:
        all_snaps[tk] = []
    all_snaps[tk].append({
        "conn_id": row.get("conn_id"),
        "t_receipt": row["t_receipt"],
        "yes_book": json.loads(row["yes_book_json"]),
        "no_book": json.loads(row["no_book_json"]),
    })

n_with_conn = sum(1 for metas in all_snaps.values() for m in metas if pd.notna(m["conn_id"]))
n_total = sum(len(metas) for metas in all_snaps.values())
print(f"Snapshot index: {len(all_snaps)} tickers, {n_total} total snapshots")
print(f"  With conn_id: {n_with_conn} ({n_with_conn/n_total:.0%})")

# How many tickers have multiple sessions?
multi = sum(1 for metas in all_snaps.values() if len(metas) > 1)
print(f"  Multi-session tickers: {multi}")

## 3. Market making simulation engine

### How the simulation works

For each ticker, we reconstruct the order book from snapshots + deltas, and simultaneously
replay the trade stream. At each point in time we know the book state (what we'd post) and
the actual trades that occurred (which would fill us).

**Strategy:** at every book update, post limit orders at the current best bid and best ask.
When a trade occurs at our posted price on the correct side, we count it as a fill.

**Fill detection logic:**
- A trade at `yes_price == our_bid` with `taker_side == "no"` means a seller hit our bid → we bought YES
- A trade at `yes_price == our_ask` with `taker_side == "yes"` means a buyer lifted our ask → we sold YES

After each fill, we measure where the mid-price goes over the next 1s, 5s, 10s, 30s, 60s
to quantify adverse selection.

**Assumptions:**
- We are always at the top of the queue (optimistic — real queue priority depends on time priority)
- We post 1 contract per side (size doesn't matter for per-contract P&L)
- We can always post at best bid/ask (no latency to detect book changes)
- Both legs are maker orders → maker fee applies

In [ ]:
def simulate_mm_ticker(ticker, all_snaps, delta_df, trades_df, window_hrs=4):
    """
    Simulate passive market making on one ticker.
    
    Reconstructs the book from all available sessions (snapshot+deltas),
    replays the trade stream, detects fills at our posted bid/ask, and
    measures adverse selection at multiple horizons.
    
    Returns a list of fill dicts with adverse selection measurements.
    """
    snaps = all_snaps.get(ticker, [])
    tk_trades = trades_df[trades_df["ticker"] == ticker].sort_values("t_receipt")
    if not snaps or tk_trades.empty:
        return []
    
    all_fills = []
    
    for meta in snaps:
        conn_id = meta.get("conn_id")
        t0 = meta["t_receipt"]
        
        # Scope deltas to this session
        if pd.notna(conn_id):
            tk_deltas = delta_df[
                (delta_df["ticker"] == ticker) &
                (delta_df["conn_id"] == conn_id) &
                (delta_df["t_receipt"] > t0)
            ].sort_values("t_receipt")
        else:
            tk_deltas = delta_df[
                (delta_df["ticker"] == ticker) &
                (delta_df["t_receipt"] > t0) &
                (delta_df["t_receipt"] < t0 + window_hrs * 3600)
            ].sort_values("t_receipt")
        
        if tk_deltas.empty:
            continue
        
        t_end = tk_deltas["t_receipt"].max()
        
        # Seed book
        yes_bk = {p: float(s) for p, s in meta["yes_book"]}
        no_bk = {p: float(s) for p, s in meta["no_book"]}
        
        # Get current book state
        stats = book_stats(yes_bk, no_bk)
        if stats[0] is None:
            continue
        cur_bid, cur_ask = stats[0], stats[1]
        cur_mid = (cur_bid + cur_ask) / 2
        
        # Trades in this session window
        session_trades = tk_trades[
            (tk_trades["t_receipt"] > t0) & (tk_trades["t_receipt"] <= t_end)
        ]
        
        # Merge deltas and trades into a single timeline
        # Process deltas to update book state, detect fills from trades
        delta_idx = 0
        delta_arr = tk_deltas[["t_receipt", "price", "delta", "side"]].values
        n_deltas = len(delta_arr)
        
        for _, trade in session_trades.iterrows():
            trade_t = trade["t_receipt"]
            
            # Apply all deltas up to this trade's timestamp
            while delta_idx < n_deltas and delta_arr[delta_idx][0] <= trade_t:
                d_price, d_delta, d_side = delta_arr[delta_idx][1], delta_arr[delta_idx][2], delta_arr[delta_idx][3]
                price_key = f"{d_price:.4f}"
                book = yes_bk if d_side == "yes" else no_bk
                apply_delta(book, price_key, d_delta)
                delta_idx += 1
            
            # Update book state
            stats = book_stats(yes_bk, no_bk)
            if stats[0] is None:
                continue
            cur_bid, cur_ask, cur_spread = stats[0], stats[1], stats[2]
            cur_bid_sz, cur_ask_sz = stats[5], stats[6]
            if cur_spread is None or cur_spread <= 0:
                continue
            cur_mid = (cur_bid + cur_ask) / 2
            
            # --- Fill detection ---
            # We post at best_bid (buy YES) and best_ask (sell YES)
            # A fill at our bid: trade.yes_price == our_bid AND taker sold YES (taker_side == "no")
            # A fill at our ask: trade.yes_price == our_ask AND taker bought YES (taker_side == "yes")
            
            trade_price = trade["yes_price"]
            taker_side = trade["taker_side"]
            fill_side = None
            fill_price = None
            
            if abs(trade_price - cur_bid) < 0.001 and taker_side == "no":
                fill_side = "buy"  # we bought YES at our bid
                fill_price = cur_bid
            elif abs(trade_price - cur_ask) < 0.001 and taker_side == "yes":
                fill_side = "sell"  # we sold YES at our ask
                fill_price = cur_ask
            
            if fill_side is None:
                continue
            
            # Record fill with book context
            fill = {
                "ticker": ticker,
                "t_fill": trade_t,
                "fill_side": fill_side,
                "fill_price": fill_price,
                "mid_at_fill": cur_mid,
                "spread_at_fill": cur_spread,
                "bid_at_fill": cur_bid,
                "ask_at_fill": cur_ask,
                "bid_depth": cur_bid_sz,
                "ask_depth": cur_ask_sz,
                "trade_count": trade["count"],
                "maker_fee": kalshi_maker_fee(fill_price),
            }
            
            # --- Adverse selection: where does mid go after fill? ---
            # Look at future trades to compute mid-price at various horizons
            for horizon_s in [1, 5, 10, 30, 60]:
                future = tk_trades[
                    (tk_trades["t_receipt"] > trade_t) &
                    (tk_trades["t_receipt"] <= trade_t + horizon_s)
                ]
                if not future.empty:
                    future_mid = (future["yes_price"].iloc[-1] + (1 - future["yes_price"].iloc[-1])) / 2
                    # Better: use last trade's yes_price as proxy for where market went
                    future_price = future["yes_price"].iloc[-1]
                    fill[f"price_{horizon_s}s"] = future_price
                else:
                    fill[f"price_{horizon_s}s"] = np.nan
            
            all_fills.append(fill)
    
    return all_fills

## 4. Run simulation across all series

Run the MM simulation on the most active tickers from each series.
Focus on tickers that have both snapshots and deltas (required for book reconstruction)
and enough trades to produce meaningful fill statistics.

In [ ]:
# --- Select tickers: top N per series by trade count, must have snapshot+deltas ---
TICKERS_PER_SERIES = 15
MIN_TRADES = 50

eligible_tickers = set(all_snaps.keys()) & set(delta_df["ticker"].unique()) & set(trades_df["ticker"].unique())
print(f"Tickers with snapshot + deltas + trades: {len(eligible_tickers)}")

# Rank by trade count within each series
selected = []
for series in ["KXNBAPTS", "KXNBAGAME", "KXNBASPREAD", "KXNBATOTAL"]:
    series_trades = trades_df[
        (trades_df["series"] == series) & (trades_df["ticker"].isin(eligible_tickers))
    ]
    top = series_trades.groupby("ticker").size().sort_values(ascending=False).head(TICKERS_PER_SERIES)
    top = top[top >= MIN_TRADES]
    selected.extend(top.index.tolist())
    print(f"  {series:15s}: {len(top)} tickers selected (top by trade count)")

print(f"\nTotal tickers to simulate: {len(selected)}")

# --- Run simulation ---
print(f"\nRunning MM simulation...")
all_fills = []
for i, ticker in enumerate(selected):
    fills = simulate_mm_ticker(ticker, all_snaps, delta_df, trades_df)
    all_fills.extend(fills)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(selected)} tickers done, {len(all_fills):,} fills so far...")

fills_df = pd.DataFrame(all_fills)
fills_df["series"] = fills_df["ticker"].apply(get_series)
fills_df["t_fill_dt"] = pd.to_datetime(fills_df["t_fill"], unit="s", utc=True)

print(f"\nSimulation complete:")
print(f"  Total fills: {len(fills_df):,}")
print(f"  Tickers with fills: {fills_df['ticker'].nunique()}")
print(f"  Buy fills:  {(fills_df['fill_side'] == 'buy').sum():,}")
print(f"  Sell fills: {(fills_df['fill_side'] == 'sell').sum():,}")
print(f"\n  Fills by series:")
for series, grp in fills_df.groupby("series"):
    print(f"    {series:15s}: {len(grp):>6,} fills across {grp['ticker'].nunique()} tickers")

## 5. Adverse selection analysis

The core question: after we get filled, does the market move against us?

For each fill, we measure **adverse selection** = how much the price moves against our
position in the seconds after the fill:

- **Buy fill**: adverse selection = `fill_price - future_price` (price drops after we buy)
- **Sell fill**: adverse selection = `future_price - fill_price` (price rises after we sell)

Positive values mean the market moved against us (bad). Negative values mean it moved
in our favor (good — but suspicious, might indicate we're not actually getting filled
at those prices in practice).

We also compute the **edge per fill** = half-spread captured minus adverse selection minus
maker fee. This is the per-contract profit if we could complete a round trip at these
prices.

In [ ]:
# --- Compute adverse selection at each horizon ---
for horizon in [1, 5, 10, 30, 60]:
    col = f"price_{horizon}s"
    # Adverse selection: positive = market moved against us
    fills_df[f"adverse_{horizon}s"] = np.where(
        fills_df["fill_side"] == "buy",
        fills_df["fill_price"] - fills_df[col],   # bought, price dropped
        fills_df[col] - fills_df["fill_price"],    # sold, price rose
    )

# Half-spread captured (what we earn by being at the inside)
fills_df["half_spread"] = fills_df["spread_at_fill"] / 2

# Edge per fill at different horizons (half-spread - adverse selection - fee)
for horizon in [1, 5, 10, 30, 60]:
    fills_df[f"edge_{horizon}s"] = (
        fills_df["half_spread"] 
        - fills_df[f"adverse_{horizon}s"] 
        - fills_df["maker_fee"]
    )

# --- Summary table ---
print("ADVERSE SELECTION BY SERIES")
print("=" * 100)
print(f"  Half-spread = what you earn per fill by posting at the inside quote.")
print(f"  Adverse selection = how much the price moves against you after a fill.")
print(f"  Edge = half_spread - adverse_selection - maker_fee (positive = profitable).\n")

for series in ["KXNBAPTS", "KXNBAGAME", "KXNBASPREAD", "KXNBATOTAL"]:
    grp = fills_df[fills_df["series"] == series]
    if grp.empty:
        continue
    
    print(f"  [{series}] — {len(grp):,} fills across {grp['ticker'].nunique()} tickers")
    print(f"    Half-spread:  median ${grp['half_spread'].median():.4f}, mean ${grp['half_spread'].mean():.4f}")
    print(f"    Maker fee:    median ${grp['maker_fee'].median():.4f}, mean ${grp['maker_fee'].mean():.4f}")
    print(f"    Spread dist:  ${grp['spread_at_fill'].quantile(0.25):.3f} / ${grp['spread_at_fill'].median():.3f} / ${grp['spread_at_fill'].quantile(0.75):.3f} (p25/p50/p75)")
    print()
    print(f"    {'Horizon':<10s}  {'Adv sel (mean)':<16s}  {'Adv sel (med)':<16s}  {'Edge (mean)':<14s}  {'Edge (med)':<14s}  {'Edge>0':<8s}  {'N'}")
    print(f"    {'-'*95}")
    for h in [1, 5, 10, 30, 60]:
        sub = grp[grp[f"adverse_{h}s"].notna()]
        if sub.empty:
            continue
        adv_mean = sub[f"adverse_{h}s"].mean()
        adv_med = sub[f"adverse_{h}s"].median()
        edge_mean = sub[f"edge_{h}s"].mean()
        edge_med = sub[f"edge_{h}s"].median()
        pct_pos = (sub[f"edge_{h}s"] > 0).mean()
        print(f"    {h:>3d}s        ${adv_mean:>14.4f}  ${adv_med:>14.4f}  ${edge_mean:>12.4f}  ${edge_med:>12.4f}  {pct_pos:>7.1%}  {len(sub):,}")
    print()

## 6. Round-trip P&L simulation

A market maker profits by completing **round trips**: buy at the bid, then sell at the ask
(or vice versa). The single-fill adverse selection above tells part of the story, but real
P&L depends on how quickly you can complete the second leg.

This section pairs consecutive buy and sell fills on the same ticker into round trips,
computing realized P&L = sell_price - buy_price - 2 * maker_fee.

We also measure **inventory duration** — how long you hold an unhedged position between
fills. Shorter = better (less exposure to directional risk).

In [ ]:
# --- Round-trip P&L: pair consecutive buy/sell fills per ticker ---

def pair_round_trips(ticker_fills):
    """
    FIFO matching: pair each buy with the next sell (or vice versa).
    Returns list of round-trip dicts.
    """
    buys = []
    sells = []
    rts = []
    
    for _, fill in ticker_fills.sort_values("t_fill").iterrows():
        if fill["fill_side"] == "buy":
            if sells:
                # Match with oldest pending sell → round trip
                sell = sells.pop(0)
                rts.append({
                    "ticker": fill["ticker"],
                    "buy_price": fill["fill_price"],
                    "sell_price": sell["fill_price"],
                    "buy_time": fill["t_fill"],
                    "sell_time": sell["t_fill"],
                    "buy_fee": fill["maker_fee"],
                    "sell_fee": sell["maker_fee"],
                    "spread_at_buy": fill["spread_at_fill"],
                    "spread_at_sell": sell["spread_at_fill"],
                    "direction": "sell_first",  # sold first, then bought to close
                })
            else:
                buys.append(fill)
        else:  # sell
            if buys:
                buy = buys.pop(0)
                rts.append({
                    "ticker": fill["ticker"],
                    "buy_price": buy["fill_price"],
                    "sell_price": fill["fill_price"],
                    "buy_time": buy["t_fill"],
                    "sell_time": fill["t_fill"],
                    "buy_fee": buy["maker_fee"],
                    "sell_fee": fill["maker_fee"],
                    "spread_at_buy": buy["spread_at_fill"],
                    "spread_at_sell": fill["spread_at_fill"],
                    "direction": "buy_first",  # bought first, then sold to close
                })
            else:
                sells.append(fill)
    
    return rts, len(buys), len(sells)  # unpaired counts

# Pair fills per ticker
all_rts = []
unpaired_buys = 0
unpaired_sells = 0

for ticker in fills_df["ticker"].unique():
    tk_fills = fills_df[fills_df["ticker"] == ticker]
    rts, ub, us = pair_round_trips(tk_fills)
    all_rts.extend(rts)
    unpaired_buys += ub
    unpaired_sells += us

rt_df = pd.DataFrame(all_rts)
if not rt_df.empty:
    rt_df["series"] = rt_df["ticker"].apply(get_series)
    rt_df["gross_pnl"] = rt_df["sell_price"] - rt_df["buy_price"]
    rt_df["total_fee"] = rt_df["buy_fee"] + rt_df["sell_fee"]
    rt_df["net_pnl"] = rt_df["gross_pnl"] - rt_df["total_fee"]
    rt_df["hold_seconds"] = abs(rt_df["sell_time"] - rt_df["buy_time"])
    rt_df["hold_minutes"] = rt_df["hold_seconds"] / 60

print("ROUND-TRIP P&L")
print("=" * 90)
print(f"  Total round trips: {len(rt_df):,}")
print(f"  Unpaired fills: {unpaired_buys} buys, {unpaired_sells} sells "
      f"({(unpaired_buys + unpaired_sells) / max(len(fills_df), 1):.0%} of fills)")

if not rt_df.empty:
    print(f"\n  Overall:")
    print(f"    Gross P&L/RT:  mean ${rt_df['gross_pnl'].mean():.4f}, median ${rt_df['gross_pnl'].median():.4f}")
    print(f"    Fees/RT:       mean ${rt_df['total_fee'].mean():.4f}")
    print(f"    Net P&L/RT:    mean ${rt_df['net_pnl'].mean():.4f}, median ${rt_df['net_pnl'].median():.4f}")
    print(f"    Win rate:      {(rt_df['net_pnl'] > 0).mean():.1%}")
    print(f"    Total net P&L: ${rt_df['net_pnl'].sum():.2f}")
    print(f"    Hold time:     median {rt_df['hold_minutes'].median():.1f} min, "
          f"mean {rt_df['hold_minutes'].mean():.1f} min")

    print(f"\n  By series:")
    print(f"  {'Series':<15s}  {'RTs':>6s}  {'Gross/RT':>10s}  {'Fee/RT':>8s}  {'Net/RT':>10s}  "
          f"{'Win%':>6s}  {'Total$':>9s}  {'Med hold':>10s}")
    print(f"  {'-'*85}")
    for series in ["KXNBAPTS", "KXNBAGAME", "KXNBASPREAD", "KXNBATOTAL"]:
        grp = rt_df[rt_df["series"] == series]
        if grp.empty:
            continue
        print(f"  {series:<15s}  {len(grp):>6,}  ${grp['gross_pnl'].mean():>9.4f}  "
              f"${grp['total_fee'].mean():>7.4f}  ${grp['net_pnl'].mean():>9.4f}  "
              f"{(grp['net_pnl'] > 0).mean():>6.1%}  ${grp['net_pnl'].sum():>8.2f}  "
              f"{grp['hold_minutes'].median():>8.1f}m")

## 7. Visualizations

### Adverse selection curves by series

How does adverse selection accumulate over time? Flat curves mean the initial fill was
fairly priced. Rising curves mean informed traders are picking us off — the longer we
hold, the more we lose.

In [ ]:
# --- Adverse selection curves by series ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
horizons = [1, 5, 10, 30, 60]

# Left: adverse selection over time
ax = axes[0]
for series in ["KXNBAPTS", "KXNBAGAME", "KXNBASPREAD", "KXNBATOTAL"]:
    grp = fills_df[fills_df["series"] == series]
    if grp.empty:
        continue
    means = [grp[f"adverse_{h}s"].mean() for h in horizons]
    # Only plot if we have data at all horizons
    if any(np.isnan(m) for m in means):
        valid = [(h, m) for h, m in zip(horizons, means) if not np.isnan(m)]
        if valid:
            ax.plot([v[0] for v in valid], [v[1] for v in valid], "o-", label=f"{series} (n={len(grp):,})")
    else:
        ax.plot(horizons, means, "o-", label=f"{series} (n={len(grp):,})")

ax.axhline(0, color="black", linestyle="-", alpha=0.3)
ax.set_xlabel("Horizon (seconds)")
ax.set_ylabel("Mean Adverse Selection ($)")
ax.set_title("Adverse Selection by Horizon")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: edge (half-spread - adverse - fee) over time
ax = axes[1]
for series in ["KXNBAPTS", "KXNBAGAME", "KXNBASPREAD", "KXNBATOTAL"]:
    grp = fills_df[fills_df["series"] == series]
    if grp.empty:
        continue
    means = [grp[f"edge_{h}s"].mean() for h in horizons]
    valid = [(h, m) for h, m in zip(horizons, means) if not np.isnan(m)]
    if valid:
        ax.plot([v[0] for v in valid], [v[1] for v in valid], "o-", label=f"{series}")

ax.axhline(0, color="red", linestyle="--", alpha=0.5, label="Break-even")
ax.set_xlabel("Horizon (seconds)")
ax.set_ylabel("Mean Edge per Fill ($)")
ax.set_title("Edge = Half-spread - Adverse Selection - Maker Fee")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Round-trip P&L distributions

Net P&L histograms per series + hold time vs P&L scatter. If the distribution has a
positive mean and fat right tail, market making is viable. If it's centered at zero or
negative, adverse selection eats the spread.

In [ ]:
# --- Round-trip P&L distributions ---
if not rt_df.empty:
    series_with_data = [s for s in ["KXNBAPTS", "KXNBAGAME", "KXNBASPREAD", "KXNBATOTAL"]
                        if not rt_df[rt_df["series"] == s].empty]
    n_series = len(series_with_data)
    
    if n_series > 0:
        fig, axes = plt.subplots(1, n_series, figsize=(5 * n_series, 4), squeeze=False)
        for i, series in enumerate(series_with_data):
            ax = axes[0, i]
            grp = rt_df[rt_df["series"] == series]
            ax.hist(grp["net_pnl"], bins=min(30, len(grp) // 2 + 1), 
                    color="steelblue", edgecolor="white", alpha=0.8)
            ax.axvline(0, color="red", linestyle="--", alpha=0.7)
            ax.axvline(grp["net_pnl"].mean(), color="green", linestyle="-",
                       label=f"mean=${grp['net_pnl'].mean():.4f}")
            ax.set_title(f"{series}\n{len(grp)} RTs, win={100*(grp['net_pnl']>0).mean():.0f}%")
            ax.set_xlabel("Net P&L ($)")
            ax.legend(fontsize=8)
        plt.suptitle("Round-Trip Net P&L Distribution", y=1.02)
        plt.tight_layout()
        plt.show()

        # Hold time vs P&L
        fig, axes = plt.subplots(1, n_series, figsize=(5 * n_series, 4), squeeze=False)
        for i, series in enumerate(series_with_data):
            ax = axes[0, i]
            grp = rt_df[rt_df["series"] == series]
            scatter = ax.scatter(grp["hold_minutes"], grp["net_pnl"], 
                                 alpha=0.4, s=15, c=grp["net_pnl"],
                                 cmap="RdYlGn", vmin=-0.05, vmax=0.05)
            ax.axhline(0, color="red", linestyle="--", alpha=0.5)
            ax.set_xlabel("Hold Time (minutes)")
            ax.set_ylabel("Net P&L ($)")
            ax.set_title(f"{series}")
        plt.suptitle("Hold Time vs Net P&L", y=1.02)
        plt.tight_layout()
        plt.show()
else:
    print("No round trips to plot.")

## 8. Spread-width conditioning: does wider spread = more profit?

Market making should be more profitable when spreads are wide (more room to capture)
and less profitable when tight. If this relationship holds, we can filter for
wide-spread opportunities only.

We also check whether fill-price level matters — fills near $0.50 have the highest
maker fees but also tend to have the most volume.

In [ ]:
# --- Conditioning: spread width and price level ---
if len(fills_df) >= 20:
    # Edge by spread bucket
    fills_df["spread_bucket"] = pd.cut(
        fills_df["spread_at_fill"], 
        bins=[0, 0.01, 0.02, 0.03, 0.05, 0.10, 1.0],
        labels=["$0.01", "$0.02", "$0.03", "$0.04-05", "$0.06-10", "$0.10+"]
    )
    
    print("EDGE BY SPREAD WIDTH (10s horizon)")
    print("=" * 80)
    print(f"  {'Spread':<12s}  {'N fills':>8s}  {'Half-spr':>10s}  {'Adv sel':>10s}  "
          f"{'Fee':>8s}  {'Edge':>10s}  {'Edge>0':>8s}")
    print(f"  {'-'*75}")
    for bucket, grp in fills_df.groupby("spread_bucket", observed=True):
        sub = grp[grp["edge_10s"].notna()]
        if len(sub) < 5:
            continue
        print(f"  {str(bucket):<12s}  {len(sub):>8,}  ${sub['half_spread'].mean():>9.4f}  "
              f"${sub['adverse_10s'].mean():>9.4f}  ${sub['maker_fee'].mean():>7.4f}  "
              f"${sub['edge_10s'].mean():>9.4f}  {(sub['edge_10s'] > 0).mean():>8.1%}")
    
    # Edge by price bucket
    fills_df["price_bucket"] = pd.cut(
        fills_df["fill_price"],
        bins=[0, 0.15, 0.30, 0.50, 0.70, 0.85, 1.0],
        labels=["$0-0.15", "$0.15-30", "$0.30-50", "$0.50-70", "$0.70-85", "$0.85-1.0"]
    )
    
    print(f"\nEDGE BY PRICE LEVEL (10s horizon)")
    print(f"  {'Price':<12s}  {'N fills':>8s}  {'Half-spr':>10s}  {'Adv sel':>10s}  "
          f"{'Fee':>8s}  {'Edge':>10s}  {'Edge>0':>8s}")
    print(f"  {'-'*75}")
    for bucket, grp in fills_df.groupby("price_bucket", observed=True):
        sub = grp[grp["edge_10s"].notna()]
        if len(sub) < 5:
            continue
        print(f"  {str(bucket):<12s}  {len(sub):>8,}  ${sub['half_spread'].mean():>9.4f}  "
              f"${sub['adverse_10s'].mean():>9.4f}  ${sub['maker_fee'].mean():>7.4f}  "
              f"${sub['edge_10s'].mean():>9.4f}  {(sub['edge_10s'] > 0).mean():>8.1%}")

    # Scatter: spread width vs edge
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ax = axes[0]
    sub = fills_df[fills_df["edge_10s"].notna()]
    ax.scatter(sub["spread_at_fill"], sub["edge_10s"], alpha=0.2, s=10)
    ax.axhline(0, color="red", linestyle="--", alpha=0.5)
    # Bin means
    bins = sub.groupby(pd.cut(sub["spread_at_fill"], bins=10))["edge_10s"].mean()
    bin_centers = [(b.left + b.right) / 2 for b in bins.index]
    ax.plot(bin_centers, bins.values, "ro-", linewidth=2, markersize=6, label="Bin mean")
    ax.set_xlabel("Spread at Fill ($)")
    ax.set_ylabel("Edge (10s) ($)")
    ax.set_title("Spread Width vs Edge")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    ax = axes[1]
    ax.scatter(sub["fill_price"], sub["edge_10s"], alpha=0.2, s=10)
    ax.axhline(0, color="red", linestyle="--", alpha=0.5)
    bins = sub.groupby(pd.cut(sub["fill_price"], bins=10))["edge_10s"].mean()
    bin_centers = [(b.left + b.right) / 2 for b in bins.index]
    ax.plot(bin_centers, bins.values, "ro-", linewidth=2, markersize=6, label="Bin mean")
    ax.set_xlabel("Fill Price ($)")
    ax.set_ylabel("Edge (10s) ($)")
    ax.set_title("Price Level vs Edge")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Not enough fills for conditioning analysis.")

## 9. Selective MM strategy: only quote when conditions are favorable

Based on the conditioning analysis, build a filtered strategy that only quotes when:
1. Spread is above a minimum threshold (wider = more room for profit)
2. Price is in a favorable range (lower fees at extremes)

Sweep the minimum-spread filter and compute net P&L for each threshold.

In [ ]:
# --- Selective strategy: sweep minimum spread threshold ---
print("SELECTIVE MM: MINIMUM SPREAD FILTER")
print("=" * 90)
print(f"  Filter fills to only those where spread_at_fill >= threshold.")
print(f"  Then pair into round trips and compute net P&L.\n")

print(f"  {'Min spread':<12s}  {'Fills':>7s}  {'RTs':>6s}  {'Gross/RT':>10s}  {'Fee/RT':>8s}  "
      f"{'Net/RT':>10s}  {'Win%':>6s}  {'Total$':>9s}  {'Sharpe':>8s}")
print(f"  {'-'*90}")

sweep_data = []
for min_spread in [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10]:
    filtered = fills_df[fills_df["spread_at_fill"] >= min_spread]
    if filtered.empty:
        continue
    
    # Re-pair round trips with filtered fills
    f_rts = []
    for ticker in filtered["ticker"].unique():
        tk_fills = filtered[filtered["ticker"] == ticker]
        rts, _, _ = pair_round_trips(tk_fills)
        f_rts.extend(rts)
    
    if not f_rts:
        continue
    
    f_rt = pd.DataFrame(f_rts)
    f_rt["gross_pnl"] = f_rt["sell_price"] - f_rt["buy_price"]
    f_rt["total_fee"] = f_rt["buy_fee"] + f_rt["sell_fee"]
    f_rt["net_pnl"] = f_rt["gross_pnl"] - f_rt["total_fee"]
    
    sharpe = f_rt["net_pnl"].mean() / f_rt["net_pnl"].std() if f_rt["net_pnl"].std() > 0 else 0
    
    print(f"  ${min_spread:<11.2f}  {len(filtered):>7,}  {len(f_rt):>6,}  "
          f"${f_rt['gross_pnl'].mean():>9.4f}  ${f_rt['total_fee'].mean():>7.4f}  "
          f"${f_rt['net_pnl'].mean():>9.4f}  {(f_rt['net_pnl'] > 0).mean():>6.1%}  "
          f"${f_rt['net_pnl'].sum():>8.2f}  {sharpe:>8.2f}")
    
    sweep_data.append({
        "min_spread": min_spread, "n_fills": len(filtered), "n_rts": len(f_rt),
        "mean_net": f_rt["net_pnl"].mean(), "total_net": f_rt["net_pnl"].sum(),
        "win_rate": (f_rt["net_pnl"] > 0).mean(), "sharpe": sharpe,
    })

# Same sweep but per-series
print(f"\n\nSELECTIVE MM BY SERIES (min spread = $0.03)")
print(f"  {'Series':<15s}  {'Fills':>7s}  {'RTs':>6s}  {'Net/RT':>10s}  {'Win%':>6s}  "
      f"{'Total$':>9s}  {'Sharpe':>8s}")
print(f"  {'-'*70}")

filtered_03 = fills_df[fills_df["spread_at_fill"] >= 0.03]
for series in ["KXNBAPTS", "KXNBAGAME", "KXNBASPREAD", "KXNBATOTAL"]:
    s_fills = filtered_03[filtered_03["series"] == series]
    if s_fills.empty:
        continue
    
    s_rts = []
    for ticker in s_fills["ticker"].unique():
        tk_fills = s_fills[s_fills["ticker"] == ticker]
        rts, _, _ = pair_round_trips(tk_fills)
        s_rts.extend(rts)
    
    if not s_rts:
        continue
    
    s_rt = pd.DataFrame(s_rts)
    s_rt["gross_pnl"] = s_rt["sell_price"] - s_rt["buy_price"]
    s_rt["total_fee"] = s_rt["buy_fee"] + s_rt["sell_fee"]
    s_rt["net_pnl"] = s_rt["gross_pnl"] - s_rt["total_fee"]
    sharpe = s_rt["net_pnl"].mean() / s_rt["net_pnl"].std() if s_rt["net_pnl"].std() > 0 else 0
    
    print(f"  {series:<15s}  {len(s_fills):>7,}  {len(s_rt):>6,}  "
          f"${s_rt['net_pnl'].mean():>9.4f}  {(s_rt['net_pnl'] > 0).mean():>6.1%}  "
          f"${s_rt['net_pnl'].sum():>8.2f}  {sharpe:>8.2f}")

## 10. Inventory-aware P&L: cumulative performance over time

The round-trip analysis above ignores timing — it pairs fills in order regardless of
how far apart they are. This section tracks cumulative P&L over time, including
mark-to-market on open positions.

For each ticker, we track:
- **Realized P&L**: locked in when a round trip completes
- **Unrealized P&L**: mark-to-market on any open position (using last trade price)
- **Position**: net contracts held (+1 = long YES, -1 = short YES, 0 = flat)